# Galaxy morphology classification with AstroLens Linformer

A tutorial-scale training example for `astrolens.models.linformer.Linformer`
(Lin et al., 2021, https://arxiv.org/abs/2110.01024), using
[`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10) —
a MultimodalUniverse-formatted copy of **Galaxy10 DECals** (17,736 galaxies,
10 discrete morphology classes, no vote-fraction preprocessing required).

The paper itself trains on the full Galaxy Zoo 2 dataset with an 8-class scheme
(round/in-between/cigar-shaped elliptical, edge-on, barred/unbarred spiral,
irregular, merger) derived from Hart et al. 2016 vote-fraction thresholds —
155,951 images, 64/16/20 split, 200 epochs. That exact label derivation isn't
reconstructable from `mwalmsley/gz2` on Hugging Face (it lacks the vote
fractions needed for the "odd feature" branch that separates irregular/merger),
and reproducing it at full scale is out of scope for a tutorial notebook. **To
reproduce the paper exactly**, use the original authors' repository and its
precomputed labels:
[`sliao-mi-luku/Galaxy-Zoo-Classification`](https://github.com/sliao-mi-luku/Galaxy-Zoo-Classification)
(`gz2_data/gz2_{train,valid,test}.csv` — galaxy ID → `label1` in 0-7, matching
the paper's split sizes almost exactly).

This notebook instead demonstrates the same architecture and training loop on
a smaller, self-contained dataset with ready-made discrete labels, split
70% train / 10% val / 20% test.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision scikit-learn

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import astrolens
from utils import gz10

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and split 70/10/20

Galaxy10 DECals' standard 10 classes (astroNN convention): disturbed, merging,
round smooth, in-between round smooth, cigar-shaped smooth, barred spiral,
unbarred tight spiral, unbarred loose spiral, edge-on without bulge, edge-on
with bulge.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
CLASS_NAMES = gz10.CLASS_NAMES
NUM_CLASSES = gz10.NUM_CLASSES

data, labels, train_idx, val_idx, test_idx = gz10.load_split_702010()

train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)

train_dataset = gz10.GZ10Dataset(data, train_idx, train_transform)
val_dataset = gz10.GZ10Dataset(data, val_idx, eval_transform)
test_dataset = gz10.GZ10Dataset(data, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset), len(test_dataset)

## Create the model

Uses `Linformer`'s default configuration, matching the paper (Lin et al., 2021):
`patch_size=28, dim=128, depth=12, heads=8, k=64`.

In [4]:
model = astrolens.create_model(
    "linformer",
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,
).to(device)

sum(p.numel() for p in model.parameters())

2790634

## Train

In [ ]:
MAX_EPOCHS = 100
LR = 3e-4
# StepLR schedule from the paper: decay LR by gamma every step_size epochs
STEP_SIZE = 5
GAMMA = 0.9

# inverse-frequency class weights from the train split, following the paper's
# use of class-weighted cross-entropy to counter GZ10's class imbalance
criterion = nn.CrossEntropyLoss(weight=gz10.class_weights(labels, train_idx).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = gz10.run_classification_epoch(
        model, train_loader, criterion, device, train=True, optimizer=optimizer
    )
    val_loss, val_acc = gz10.run_classification_epoch(model, val_loader, criterion, device, train=False)
    scheduler.step()
    print(
        f"epoch {epoch}/{MAX_EPOCHS} "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f} "
        f"lr={scheduler.get_last_lr()[0]:.2e}"
    )

## Evaluate on the held-out test split

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_acc=0.772 test_f1_macro=0.752

                         precision    recall  f1-score   support

              disturbed       0.43      0.49      0.46       216
                merging       0.82      0.83      0.83       371
           round_smooth       0.93      0.88      0.90       529
in_between_round_smooth       0.84      0.92      0.88       405
    cigar_shaped_smooth       0.53      0.78      0.63        67
          barred_spiral       0.82      0.78      0.80       409
  unbarred_tight_spiral       0.60      0.76      0.67       366
  unbarred_loose_spiral       0.67      0.52      0.59       525
       edge_on_no_bulge       0.88      0.87      0.87       285
     edge_on_with_bulge       0.92      0.86      0.89       375

               accuracy                           0.77      3548
              macro avg       0.75      0.77      0.75      3548
           weighted avg       0.78      0.77      0.77      3548



## Next steps

- Raise `MAX_EPOCHS` or add a learning-rate schedule / early stopping for a longer run.
- To reproduce the paper's actual 8-class Galaxy Zoo 2 result (155,951 images,
  200 epochs, class-weighted loss), follow
  [`sliao-mi-luku/Galaxy-Zoo-Classification`](https://github.com/sliao-mi-luku/Galaxy-Zoo-Classification)
  directly — it ships the precomputed `label1` splits this notebook doesn't
  attempt to rederive.